In [2]:
import os
import shutil


In [3]:
data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

# Iteriere über alle Ordner im aktuellen Verzeichnis
for root, dirs, files in os.walk('.'):
    for dir_name in dirs:
        # Überprüfen, ob "batch" im Ordnernamen enthalten ist
        if 'batch' in dir_name:
            batch_folder_path = os.path.join(root, dir_name)
            # Durchsuchen der Dateien in diesem Ordner
            for file_name in os.listdir(batch_folder_path):
                # Überprüfen, ob die Datei eine .txt oder .ann Datei ist
                if file_name.endswith('.txt') or file_name.endswith('.ann'):
                    file_path = os.path.join(batch_folder_path, file_name)
                    shutil.move(file_path, data_folder)
                    print(f"Moved {file_path} to {data_folder}")

Moved .\LCT\batch1\NCT03860038.ann to data
Moved .\LCT\batch1\NCT03860038.txt to data
Moved .\LCT\batch1\NCT03860415.ann to data
Moved .\LCT\batch1\NCT03860415.txt to data
Moved .\LCT\batch1\NCT03860428.ann to data
Moved .\LCT\batch1\NCT03860428.txt to data
Moved .\LCT\batch1\NCT03861078.ann to data
Moved .\LCT\batch1\NCT03861078.txt to data
Moved .\LCT\batch1\NCT03861130.ann to data
Moved .\LCT\batch1\NCT03861130.txt to data
Moved .\LCT\batch1\NCT03861286.ann to data
Moved .\LCT\batch1\NCT03861286.txt to data
Moved .\LCT\batch1\NCT03861468.ann to data
Moved .\LCT\batch1\NCT03861468.txt to data
Moved .\LCT\batch1\NCT03861520.ann to data
Moved .\LCT\batch1\NCT03861520.txt to data
Moved .\LCT\batch1\NCT03861858.ann to data
Moved .\LCT\batch1\NCT03861858.txt to data
Moved .\LCT\batch1\NCT03861910.ann to data
Moved .\LCT\batch1\NCT03861910.txt to data
Moved .\LCT\batch1\NCT03862001.ann to data
Moved .\LCT\batch1\NCT03862001.txt to data
Moved .\LCT\batch1\NCT03862014.ann to data
Moved .\LCT

## Erzeuge IC und EC getrennt

In [10]:
import os
import re

def read_criteria_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

def write_to_file(file_path, content):
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(content)

def process_files(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    unsplit_files = []

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            nct_number = re.findall(r'NCT\d+', filename)
            if not nct_number:
                continue
            nct_number = nct_number[0]
            file_path = os.path.join(input_folder, filename)
            content = read_criteria_file(file_path)

            lines = content.split('\n')
            inclusion_criteria = []
            exclusion_criteria = []
            current_section = None

            for line in lines:
                line_lower = line.lower().strip()
                print(line_lower)
                if "inclusion criteria" in line_lower:
                    current_section = "inclusion"
                    continue
                elif "exclusion criteria" in line_lower:
                    current_section = "exclusion"
                    continue

                if current_section == "inclusion":
                    if line.strip().startswith("-") or re.match(r'^\s*\d+\.\s', line.strip()):
                        inclusion_criteria.append(line.strip())
                    elif line.strip():
                        inclusion_criteria.append(line.strip())
                elif current_section == "exclusion":
                    if line.strip().startswith("-") or re.match(r'^\s*\d+\.\s', line.strip()):
                        exclusion_criteria.append(line.strip())
                    elif line.strip():
                        exclusion_criteria.append(line.strip())

            # Remove empty lines and duplicates
            inclusion_criteria = list(dict.fromkeys([line for line in inclusion_criteria if line]))
            exclusion_criteria = list(dict.fromkeys([line for line in exclusion_criteria if line]))

            if not inclusion_criteria or not exclusion_criteria:
                unsplit_files.append(filename)

            if inclusion_criteria:
                inc_filename = f"{nct_number}_inc.txt"
                inc_file_path = os.path.join(output_folder, inc_filename)
                write_to_file(inc_file_path, "\n".join(inclusion_criteria))

            if exclusion_criteria:
                exc_filename = f"{nct_number}_exc.txt"
                exc_file_path = os.path.join(output_folder, exc_filename)
                write_to_file(exc_file_path, "\n".join(exclusion_criteria))

    # Print out the filenames that couldn't be split into inc and exc
    if unsplit_files:
        print("Files that couldn't be split into inclusion and exclusion criteria:")
        for file in unsplit_files:
            print(file)

# Beispielaufruf
input_folder = "data_parsed_1_fix"
output_folder = "data_parsed_1_half"
process_files(input_folder, output_folder)


inclusion criteria:
1. inflammatory bowel disease
2. on methotrexate at appropriate dosing
3. normal folate levels at onset of study
4. treatment with folic acid
5. ages 2-21 years
exclusion criteria:
1. abnormal folate levels
2. age > 21  [or] or less than 2
inclusion criteria:
-  patients with one or more events of hip dislocation after a primary tha
-  controls with a primary tha [not] without events of hip dislocation
exclusion criteria:
-
inclusion criteria:
1. age ≥ 18, male  [or] or female;
2. subject must have had documented mm;
3. at screening phase, subject must have measurable disease;
4. subject is in a state of progressive disease (pd);
5. subject must have life expectancy of no less than 6 months;
6. subject must have an ecog (eastern cooperative oncology group) performance status score of 0~2;
exclusion criteria:
1. subject has received anti-cd38 monoclonal antibody treatment previously;
2. subject has received car-t cell therapy previously;
3. subject has previously rec

In [6]:
# NCT03860181

## Zähle Operatoren

In [4]:
import os
import json

def count_operators(directory):
    and_count = 0
    or_count = 0
    not_count = 0

    json_files = [f for f in os.listdir(directory) if f.endswith('.json')]

    for file in json_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for entry in data:
                if entry['type'] == 'And':
                    and_count += 1
                elif entry['type'] == 'Or':
                    or_count += 1
                elif entry['type'] == 'Negation':
                    not_count += 1

    return and_count, or_count, not_count

directory = 'all_entitys'
and_count, or_count, not_count = count_operators(directory)

print(f"Total AND operators: {and_count}")
print(f"Total OR operators: {or_count}")
print(f"Total NOT operators: {not_count}")


Total AND operators: 819
Total OR operators: 4153
Total NOT operators: 948


In [6]:
986*2

1972